# Init

In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

from imblearn.over_sampling import SMOTE

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score

from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, recall_score

from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/models/grupos-x-submissions-x-char.csv'


In [ ]:
file = pd.read_csv(file_path)


In [ ]:
file.columns

In [ ]:
file[['outcome_service_url',
       'result', 'status', 'submitted_on', 'taskid', 'text', 'url',
       'archivo_origen', 'solo_fecha', 'ano', 'mes', 'dondeconocioelcurso',
       'motivacion', 'ocupacion', 'lenguajes_programacion',
       'nivelprogramacion', 'dedicacionsemanal', 'razon_realizar_curso',
       'otra_motivacion', 'otra_ocupacion', 'email', 'name', 'uncode_usr',
       'conteo_lenguajes', 'otro_lenguaje', 'otro_medio']].head()

In [ ]:
df_for_model = file[[
       'username', 'platform',
       'actividad', 'submission_activity_user', 'max_grade',
       'custom[custom_summary_result]', 'grade', 'peso',
       'result', 'status', 'best',
       'dias_totales', 'dondeconocioelcurso','conteo_lenguajes',
       'motivacion', 'ocupacion', 'lenguajes_programacion',
       'nivelprogramacion', 'dedicacionsemanal',
       'otra_motivacion', 'otra_ocupacion',
       'otro_lenguaje', 'otro_medio', 'grupo', 'dias_x_actividad']]

In [ ]:
df_for_model['grupo'].value_counts()

Columnas no tenidas en cuenta

- 'url',
- '_id',
- 'outcome_service_url',
- 'mes',
- 'name',
- 'custom[custom_additional_info]',
- 'is_late_submission',
- 'email',
- 'platform',
- 'text',
-'solo_fecha',
-'outcome_result_id',
-'submitted_on',
-'ano',
- 'nota_final',
- 'fecha_inicio',
- 'razon_realizar_curso',
- 'username',
- 'outcome_consumer_key',
- 'archivo_origen',
- 'input',
- 'fecha_final',
- 'archive',
- 'prefijo',
- 'uncode_usr',
- 'taskid',
- 'nota_ponderada'

In [ ]:
columnas_unicas_df1 = set(file.columns) - set(df_for_model.columns)
print(f"Columnas solo en file: {columnas_unicas_df1}")

In [ ]:
df_for_model.head(5)

In [ ]:
print(df_for_model.dtypes)

# Creación de variables dummy a partir de variables de texto

## Otro_lenguaje

Dado el análisis de bigramas y de palabras más repetidas hecho anteriormente, esta columna tiene las siguientes categorias posibles:

basic: 28
visual: 24
html: 13
matlab: 12
css: 9
sql: 8
php: 7
pascal: 6
arduino: 5
aprender: 5
excel: 5
vba: 5
typescript: 3
lenguajes: 3
basico: 3

In [ ]:
df_to_test = df_for_model.copy()

perfiles = {
    'pref_legacy': ['basic', 'visual', 'pascal'],
    'pref_web_sql': ['html', 'css', 'sql', 'php', 'typescript'],
    'pref_ingenieria': ['matlab', 'arduino'],
    'pref_office': ['excel', 'vba'],
    'pref_principiante': ['aprender', 'basico']
}

def categorizar_lenguajes(df):
    col_texto = df['otro_lenguaje'].fillna('').str.lower()

    for nombre_perfil, palabras in perfiles.items():
        df[nombre_perfil] = col_texto.apply(
            lambda x: 1 if any(palabra in x for palabra in palabras) else 0
        )
    return df

df_to_test = categorizar_lenguajes(df_to_test)

In [ ]:
df_for_model.columns

In [ ]:

columnas_perfiles = ['pref_legacy', 'pref_web_sql', 'pref_ingenieria', 'pref_office', 'pref_principiante']

df_to_test['no_mapeado'] = (df_to_test[columnas_perfiles].sum(axis=1) == 0)

total_no_mapeados = df_to_test['no_mapeado'].sum()
porcentaje = (total_no_mapeados / len(df_to_test)) * 100

print(f"Usuarios no identificados: {total_no_mapeados} ({porcentaje:.2f}%)")

Se estudia qué ocurrió con estos que no se mapearon

In [ ]:
revisar = df_to_test[df_to_test['no_mapeado'] == True]['otro_lenguaje'].unique()

print("Ejemplos de textos que no se mapearon:")
print(revisar[:20])

In [ ]:
no_mapeados_con_texto = df_to_test[df_to_test['no_mapeado'] & df_to_test['otro_lenguaje'].notna()]
solo_vacios = df_to_test[df_to_test['no_mapeado'] & df_to_test['otro_lenguaje'].isna()]

print(f"No mapeados que tienen texto: {len(no_mapeados_con_texto)}")
print(f"No mapeados que están vacíos (NaN): {len(solo_vacios)}")

Muchos son campos que vienen NaN

In [ ]:
from collections import Counter
import re

texto_restante = df_to_test[(df_to_test['no_mapeado'] == True) & (df_to_test['otro_lenguaje'].notna())]['otro_lenguaje']

texto_unido = " ".join(texto_restante.astype(str).str.lower())

palabras = re.findall(r'\b\w{3,}\b', texto_unido)

print("Top 20 palabras 'perdidas':")
for palabra, count in Counter(palabras).most_common(20):
    print(f"{palabra}: {count}")

In [ ]:
df_for_model_v2 = df_for_model.copy()

perfiles = {
    'pref_legacy': ['basic', 'visual', 'pascal', 'cobol', 'fortran'],
    'pref_web_sql': ['html', 'css', 'sql', 'php', 'typescript', 'javascript'],
    'pref_bi_datos': ['dax', 'power', 'bi'],
    'pref_academico': ['latex', 'overleaf', 'matlab'],
    'pref_tecnico_otros': ['gdscript', 'arduino', 'proton', 'python'],
}

def categorizar_final(texto):
    if texto in ['declaro_sin_experiencia', 'campo_corregido']:
        return texto

    if pd.isna(texto):
        return 'no_reporta'

    t = str(texto).lower().strip()


    if len(t) < 2 or t in ['...', '---', '***', 'xyz', 'asdf', 'hhhhh']:
        return 'es_spam_ruido'

    for nombre_perfil, palabras in perfiles.items():
        if any(p in t for p in palabras):
            return nombre_perfil

    return 'otros_lenguajes_especificos'

df_for_model_v2['perfil_tecnico_final'] = df_for_model_v2['otro_lenguaje'].apply(categorizar_final)

In [ ]:
df_for_model_v2.columns

In [ ]:
print(df_for_model_v2['perfil_tecnico_final'].value_counts())

info_duplicada: se realizó un intercambio entre la columna principal y la de "otro" y se marca el registro como tal, ya que no es que la persona no haya incluido algo allí sino que llenó mal el formulario.

In [ ]:
df_for_model_v2 = pd.get_dummies(df_for_model_v2, columns=['perfil_tecnico_final'], prefix='otro_lenguaje')


In [ ]:
df_for_model_v2.columns

In [ ]:
df_for_model_v2 = df_for_model_v2[[
       'username', 'platform', 'actividad', 'submission_activity_user',
       'max_grade', 'custom[custom_summary_result]', 'grade', 'peso', 'result',
       'status', 'best', 'dias_x_actividad', 'dondeconocioelcurso',
       'conteo_lenguajes', 'motivacion', 'ocupacion', 'lenguajes_programacion',
       'nivelprogramacion', 'dedicacionsemanal', 'otra_motivacion',
       'otra_ocupacion', #quitamos otro_lenguaje
       'otro_medio', 'grupo',
       'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql']]

## Donde conoció el curso y Otro_medio

In [ ]:
df_for_model_v2['otro_medio'].value_counts()

In [ ]:
df_for_model_v3 = df_for_model_v2.copy()

def categorizar_otro_medio(texto):
    if texto in ['campo_corregido', 'declaro_sin_experiencia']:
        return texto

    if pd.isna(texto):
        return 'no_reporta'

    t = str(texto).lower().strip()

    if any(p in t for p in ['profesor', 'universidad', 'materia', 'clase', 'docente', 'unal', 'nacional']):
        return 'origen_academico'

    if any(p in t for p in ['facebook', 'tik', 'tok', 'social']):
        return 'origen_redes_sociales'

    if any(p in t for p in ['pagina', 'web', 'internet', 'busqueda']):
        return 'origen_web_busqueda'

    if len(t) < 3:
        return 'es_spam_ruido'

    return 'otros_medios_especificos'

df_for_model_v3['perfil_medio_final'] = df_for_model_v3['otro_medio'].apply(categorizar_otro_medio)

In [ ]:
df_for_model_v3['perfil_medio_final'].value_counts()

In [ ]:
def consolidar_canal(row):
    if row['dondeconocioelcurso'] == 'otro':
        return row['perfil_medio_final']
    return row['dondeconocioelcurso']

df_for_model_v3['canal_consolidado'] = df_for_model_v3.apply(consolidar_canal, axis=1)

In [ ]:
df_for_model_v3['canal_consolidado'].value_counts()

In [ ]:
map_canales = {
    'recomendacion': 'origen_social_directo',
    'origen_academico': 'canal_social_directo',
    'facebook': 'canal_ads_social',
    'instagram': 'canal_ads_social',
    'origen_redes_sociales': 'canal_ads_social',
    'google': 'canal_busqueda',
    'pagina web': 'canal_busqueda',
    'origen_web_busqueda': 'canal_busqueda',
    'email': 'canal_push_notif',
    'linkedin': 'canal_push_notif',
    'sms': 'canal_push_notif'
}

df_for_model_v3['canal_grupo'] = df_for_model_v3['canal_consolidado'].map(map_canales)

df_canal_dummies = pd.get_dummies(df_for_model_v3['canal_grupo'], prefix='medio', dtype=int)
df_for_model_v3 = pd.concat([df_for_model_v3, df_canal_dummies], axis=1)

In [ ]:

df_medios_dummies = pd.get_dummies(df_for_model_v3['perfil_medio_final'], prefix='medio')

df_medios_dummies = df_medios_dummies.astype(int)

df_for_model_v3 = pd.concat([df_for_model_v3, df_medios_dummies], axis=1)

print("Nuevas columnas de medios creadas:")
print(df_medios_dummies.columns.tolist())

In [ ]:
df_for_model_v3.columns

In [ ]:
df_for_model_v3 = df_for_model_v3[['username', 'platform', 'actividad', 'submission_activity_user',
       'max_grade', 'custom[custom_summary_result]', 'grade', 'peso', 'result',
       'status', 'best', 'dias_x_actividad',
       'conteo_lenguajes', 'motivacion', 'ocupacion', 'lenguajes_programacion',
       'nivelprogramacion', 'dedicacionsemanal', 'otra_motivacion',
       'otra_ocupacion', #se quita otro_medio y dondeconocioel curso
       'grupo',
       'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql',
        #
       'medio_canal_ads_social', 'medio_canal_busqueda',
       'medio_canal_push_notif', 'medio_canal_social_directo',
       'medio_campo_corregido', 'medio_declaro_sin_experiencia',
       'medio_es_spam_ruido', 'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos']]

## Motivación y Otra_motivacion

In [ ]:
df_for_model_v3['otra_motivacion'].value_counts()

In [ ]:
df_for_model_v4 = df_for_model_v3.copy()

def categorizar_motivacion(texto):
    if texto in ['campo_corregido', 'declaro_sin_experiencia']:
        return texto
    if pd.isna(texto):
        return 'no_reporta'

    t = str(texto).lower().strip()

    if any(p in t for p in ['ciencia datos', 'machine learning', 'inteligencia artificial', 'analisis datos', 'data science']):
        return 'mot_ciencia_datos'

    if any(p in t for p in ['desarrollo web', 'desarrollo software', 'backend', 'programar', 'desarrollo']):
        return 'mot_desarrollo'

    if any(p in t for p in ['carrera profesional', 'automatizar', 'procesos', 'laboral', 'trabajo', 'empleo']):
        return 'mot_profesional'

    if any(p in t for p in ['aprender', 'conocimientos', 'curiosidad', 'mundo', 'quiero']):
        return 'mot_aprendizaje_general'

    if len(t) < 4:
        return 'es_spam_ruido'

    return 'otras_motivaciones_especificas'

df_for_model_v4['perfil_motivacion_final'] = df_for_model_v4['otra_motivacion'].apply(categorizar_motivacion)

In [ ]:
df_for_model_v4['perfil_motivacion_final'].value_counts()

In [ ]:
df_for_model_v4.columns

Se consolidará motivación y otra_motivación

In [ ]:
map_mot_principal = {
    'quiero aprender a programar computadores en general (es mi primer lenguaje de programación)': 'principiante_general',
    'quiero entrar al mundo de machine learning y ciencia de datos': 'data_science_ia',
    'quiero saber acerca de desarrollo backend': 'dev_backend',
    'quiero involucrarme en desarrollo web': 'dev_web',
    'otro': 'otro_especificado'
}

df_for_model_v4['motivacion_short'] = df_for_model_v4['motivacion'].map(map_mot_principal)

In [ ]:
def consolidar_motivacion(row):
    if row['motivacion_short'] == 'otro_especificado':
        return row['perfil_motivacion_final']
    return row['motivacion_short']

df_for_model_v4['motivacion_consolidada'] = df_for_model_v4.apply(consolidar_motivacion, axis=1)

In [ ]:
df_mot_dummies = pd.get_dummies(df_for_model_v4['motivacion_consolidada'], prefix='mot', dtype=int)

df_for_model_v4 = pd.concat([df_for_model_v4, df_mot_dummies], axis=1)

print("Categorías finales de motivación listas para el modelo:")
print(df_mot_dummies.columns.tolist())

In [ ]:
df_for_model_v4.columns

In [ ]:
df_for_model_v4 = df_for_model_v4[
    [
       'username', 'platform', 'actividad', 'submission_activity_user',
       'max_grade', 'custom[custom_summary_result]', 'grade', 'peso', 'result',
       'status', 'best', 'dias_x_actividad',
       'conteo_lenguajes',
       # se elimina columna motivacion
       'ocupacion', 'lenguajes_programacion',
       'nivelprogramacion', 'dedicacionsemanal',
       # se elimina columna otra_motivacion
       'otra_ocupacion', 'grupo', 'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql',
       ##
       'medio_canal_ads_social', 'medio_canal_busqueda',
       'medio_canal_push_notif', 'medio_canal_social_directo',
       'medio_campo_corregido', 'medio_declaro_sin_experiencia',
       'medio_es_spam_ruido', 'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos',
       ##
       'mot_data_science_ia',
       'mot_declaro_sin_experiencia', 'mot_dev_backend', 'mot_dev_web',
       'mot_es_spam_ruido', 'mot_mot_aprendizaje_general',
       'mot_mot_ciencia_datos', 'mot_mot_desarrollo', 'mot_mot_profesional',
       'mot_no_reporta', 'mot_otras_motivaciones_especificas',
       'mot_principiante_general'
    ]
]

## Ocupación y Otra ocupación

In [ ]:
df_for_model_v5 = df_for_model_v4.copy()
def categorizar_ocupacion(texto):

    if texto == 'campo_corregido':
        return 'perfil_validado'

    if pd.isna(texto):
        return 'no_reporta'

    t = str(texto).lower().strip()

    if any(p in t for p in ['docente', 'profesor', 'maestro', 'educador']):
        return 'perfil_educador'

    if any(p in t for p in ['desempleado', 'cesante', 'buscando', 'sin empleo']):
        return 'perfil_busqueda_empleo'

    if 'estudio' in t and ('trabajo' in t or 'empleo' in t):
        return 'perfil_estudia_trabaja'

    if len(t) < 4 or any(p in t for p in ['asdf', 'aaaa', 'jjjj', 'gimna', 'prueba']):
        return 'es_spam_ruido'

    return 'otros_perfiles_ocupacion'

df_for_model_v5['perfil_ocupacion_final'] = df_for_model_v5['otra_ocupacion'].apply(categorizar_ocupacion)

In [ ]:
print(df_for_model_v5['perfil_ocupacion_final'].value_counts())

 Se consolida columna ocupación con otra_ocupacion

In [ ]:
print(df_for_model_v5['ocupacion'].value_counts())

In [ ]:
def consolidar_ocupacion(row):
    if row['ocupacion'] == 'otro':
        return row['perfil_ocupacion_final']
    return row['ocupacion']

df_for_model_v5['ocupacion_consolidada'] = df_for_model_v5.apply(consolidar_ocupacion, axis=1)

In [ ]:
df_ocu_dummies = pd.get_dummies(df_for_model_v5['ocupacion_consolidada'], prefix='ocupacion', dtype=int)

df_for_model_v5 = pd.concat([df_for_model_v5, df_ocu_dummies], axis=1)

print("Columnas de ocupación listas:")
print(df_ocu_dummies.columns.tolist())

In [ ]:
df_for_model_v5.columns

In [ ]:
df_for_model_v5 = df_for_model_v5[
    ['username', 'platform', 'actividad', 'submission_activity_user',
       'max_grade', 'custom[custom_summary_result]', 'grade', 'peso', 'result',
       'status', 'best', 'dias_x_actividad',
       'conteo_lenguajes',
       # se elimina columna ocupacion
       'lenguajes_programacion',
       'nivelprogramacion', 'dedicacionsemanal',
       'grupo',
        # se elimina columan de otra_ocupacion
       'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql',
       ###
       'medio_canal_ads_social', 'medio_canal_busqueda',
       'medio_canal_push_notif', 'medio_canal_social_directo',
       'medio_campo_corregido', 'medio_declaro_sin_experiencia',
       'medio_es_spam_ruido', 'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos',
       ##
       'mot_data_science_ia',
       'mot_declaro_sin_experiencia', 'mot_dev_backend', 'mot_dev_web',
       'mot_es_spam_ruido', 'mot_mot_aprendizaje_general',
       'mot_mot_ciencia_datos', 'mot_mot_desarrollo', 'mot_mot_profesional',
       'mot_no_reporta', 'mot_otras_motivaciones_especificas',
       'mot_principiante_general',
       ##
       'ocupacion_empleado',
       'ocupacion_es_spam_ruido', 'ocupacion_estudiante',
       'ocupacion_independiente', 'ocupacion_no_reporta',
       'ocupacion_otros_perfiles_ocupacion', 'ocupacion_pensionado',
       'ocupacion_perfil_busqueda_empleo', 'ocupacion_perfil_educador',
       'ocupacion_perfil_estudia_trabaja', 'ocupacion_tareas_del_hogar']
]

## Dedicación semanal

In [ ]:
print(df_for_model_v5['dedicacionsemanal'].value_counts())

In [ ]:
map_dedicacion = {
    '1-2 horas': 0,
    '3-4 horas': 1,
    '5-6 horas': 2,
    '7 horas o más': 3
}

df_for_model_v5['dedicacion_num'] = df_for_model_v5['dedicacionsemanal'].map(map_dedicacion)

df_for_model_v5['dedicacion_num'] = df_for_model_v5['dedicacion_num'].fillna(-1).astype(int)

In [ ]:
df_for_model_v5.columns

In [ ]:
df_for_model_v5 = df_for_model_v5 [
    ['username', 'platform', 'actividad', 'submission_activity_user',
       'max_grade', 'custom[custom_summary_result]', 'grade', 'peso', 'result',
       'status', 'best', 'dias_x_actividad',
       'conteo_lenguajes', 'lenguajes_programacion', 'nivelprogramacion',
       ## se elimina 'dedicacionsemanal',
       'grupo',
       # lenguajes
       'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql',
       # medios
       'medio_canal_ads_social', 'medio_canal_busqueda',
       'medio_canal_push_notif', 'medio_canal_social_directo',
       'medio_campo_corregido', 'medio_declaro_sin_experiencia',
       'medio_es_spam_ruido', 'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos',
       # motivaciones
       'mot_data_science_ia',
       'mot_declaro_sin_experiencia', 'mot_dev_backend', 'mot_dev_web',
       'mot_es_spam_ruido', 'mot_mot_aprendizaje_general',
       'mot_mot_ciencia_datos', 'mot_mot_desarrollo', 'mot_mot_profesional',
       'mot_no_reporta', 'mot_otras_motivaciones_especificas',
       'mot_principiante_general',
       # ocupaciones
       'ocupacion_empleado',
       'ocupacion_es_spam_ruido', 'ocupacion_estudiante',
       'ocupacion_independiente', 'ocupacion_no_reporta',
       'ocupacion_otros_perfiles_ocupacion', 'ocupacion_pensionado',
       'ocupacion_perfil_busqueda_empleo', 'ocupacion_perfil_educador',
       'ocupacion_perfil_estudia_trabaja', 'ocupacion_tareas_del_hogar',
       # dedicacion
       'dedicacion_num']
]

## Nivel de programación

In [ ]:
print(df_for_model_v4['nivelprogramacion'].value_counts())

In [ ]:
map_nivel = {
    'nulo': 0,
    'basico': 1,
    'intermedio': 2,
    'avanzado': 3
}

df_for_model_v5['nivel_num'] = df_for_model_v5['nivelprogramacion'].map(map_nivel)

df_for_model_v5['nivel_num'] = df_for_model_v5['nivel_num'].fillna(-1).astype(int)

In [ ]:
df_for_model_v5['nivel_num'].value_counts()

In [ ]:
df_for_model_v5 = df_for_model_v5 [
    [ 'username', 'platform',
      ## actividades
       'actividad', 'submission_activity_user',
       'max_grade', 'custom[custom_summary_result]', 'grade', 'peso', 'result',
       'status', 'best', 'dias_x_actividad',
      ### --- caracterizacion
       'conteo_lenguajes', 'lenguajes_programacion',
       ## se elimina 'nivelprogramacion',
       'grupo',
       # lenguajes
       'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql',
       # medios
       'medio_canal_ads_social', 'medio_canal_busqueda',
       'medio_canal_push_notif', 'medio_canal_social_directo',
       'medio_campo_corregido', 'medio_declaro_sin_experiencia',
       'medio_es_spam_ruido', 'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos',
       # motivaciones
       'mot_data_science_ia',
       'mot_declaro_sin_experiencia', 'mot_dev_backend', 'mot_dev_web',
       'mot_es_spam_ruido', 'mot_mot_aprendizaje_general',
       'mot_mot_ciencia_datos', 'mot_mot_desarrollo', 'mot_mot_profesional',
       'mot_no_reporta', 'mot_otras_motivaciones_especificas',
       'mot_principiante_general',
       # ocupaciones
       'ocupacion_empleado',
       'ocupacion_es_spam_ruido', 'ocupacion_estudiante',
       'ocupacion_independiente', 'ocupacion_no_reporta',
       'ocupacion_otros_perfiles_ocupacion', 'ocupacion_pensionado',
       'ocupacion_perfil_busqueda_empleo', 'ocupacion_perfil_educador',
       'ocupacion_perfil_estudia_trabaja', 'ocupacion_tareas_del_hogar',
       # dedicacion
       'dedicacion_num',
       # nivel programacion
       'nivel_num']
]

In [ ]:
df_final_for_model = df_for_model_v5

# MODELO DE CLASIFICACIÓN DE CARACTERIZACIÓN

In [ ]:
df_char_model = df_for_model_v5 [
    [ 'username', 'platform',
      ### --- caracterizacion
       'conteo_lenguajes',
       ## se elimina 'nivelprogramacion',
       'grupo',
       # lenguajes
       'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql',
       # medios
       'medio_canal_ads_social', 'medio_canal_busqueda',
       'medio_canal_push_notif', 'medio_canal_social_directo',
       'medio_campo_corregido', 'medio_declaro_sin_experiencia',
       'medio_es_spam_ruido', 'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos',
       # motivaciones
       'mot_data_science_ia',
       'mot_declaro_sin_experiencia', 'mot_dev_backend', 'mot_dev_web',
       'mot_es_spam_ruido', 'mot_mot_aprendizaje_general',
       'mot_mot_ciencia_datos', 'mot_mot_desarrollo', 'mot_mot_profesional',
       'mot_no_reporta', 'mot_otras_motivaciones_especificas',
       'mot_principiante_general',
       # ocupaciones
       'ocupacion_empleado',
       'ocupacion_es_spam_ruido', 'ocupacion_estudiante',
       'ocupacion_independiente', 'ocupacion_no_reporta',
       'ocupacion_otros_perfiles_ocupacion', 'ocupacion_pensionado',
       'ocupacion_perfil_busqueda_empleo', 'ocupacion_perfil_educador',
       'ocupacion_perfil_estudia_trabaja', 'ocupacion_tareas_del_hogar',
       # dedicacion
       'dedicacion_num',
       # nivel programacion
       'nivel_num']
]

In [ ]:
user_counts = df_char_model.groupby(['username', 'platform']).size().reset_index(name='count')

In [ ]:
df_char_model_unique = df_char_model.drop_duplicates(subset=['username', 'platform'], keep='first')

In [ ]:
len(df_char_model_unique)

In [ ]:
print(df_char_model_unique['grupo'].value_counts())

In [ ]:
len(user_counts)

## Con tres clases SMOTE

In [ ]:
exclude = ['grupo', 'username', 'platform']
X = df_char_model_unique.drop(columns=exclude)
y = df_char_model_unique['grupo']

X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Distribución original: {y_train.value_counts().to_dict()}")
print(f"Distribución con SMOTE: {y_train_res.value_counts().to_dict()}")


### Búsqueda de hiperparámetros para Random Forest, tres clases, balanceadas

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': [None]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_res, y_train_res)

print(f"Mejores parámetros: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_

In [ ]:
y_pred_best = best_rf.predict(X_test)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best))

In [ ]:
importancias_char = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importancias_char.head(10), palette='magma')

plt.title('Top 10 Predictores: Caracterización Inicial (Día 0)', fontsize=14)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable de la Encuesta', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

print(importancias_char.head(10))

Regresión lineal, ver el p valor.

### Búsqueda de hiperparámetros para SVM

In [ ]:

param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'poly', 'sigmoid'],
    'class_weight': [None]
}

grid_svm = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=param_grid_svm,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_svm.fit(X_train_res, y_train_res)

print(f"Mejores parámetros SVM: {grid_svm.best_params_}")
best_svm = grid_svm.best_estimator_

In [ ]:
y_pred_best_svm = grid_svm.predict(X_test)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best_svm))

## Con dos clases

### Búsqueda de hiperparámetros para Random Forest, dos clases

Se intenta otro enfoque ya que los resultados no son muy buenos, juntando los desertores con reprobados para hacer un clasificador binario

In [ ]:
# Aprobados y Reprobados se agrupan
df_char_model_unique['grupo_binario'] = df_char_model_unique['grupo'].replace({
    'desertores': 'reprobados',
    'reprobados': 'reprobados'
})


In [ ]:
print(df_char_model_unique['grupo_binario'].value_counts())

In [ ]:
y_bin = df_char_model_unique['grupo_binario']
X_bin = df_char_model_unique.drop(columns=['username', 'platform', 'grupo', 'grupo_binario'])

se observa cómo va la búsqueda SIN balancear estas dos clases

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', None]
}

X_train, X_test, y_train, y_test = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin)




In [ ]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_

In [ ]:
y_pred_best = best_rf.predict(X_test)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best))

PENDIENTE - XGBOOST


In [ ]:
importancias_char = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importancias_char.head(10), palette='magma')

plt.title('Top 10 Predictores: Caracterización Inicial (Día 0)', fontsize=14)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable de la Encuesta', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

print(importancias_char.head(10))

No es tan interpretable por los resultados

### Búsqueda de hiperparámetros para SVM, dos clases.

In [ ]:
param_grid_light = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.01],
    'kernel': ['rbf', 'linear']
}

grid_svm = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=param_grid_light,
    scoring='recall_macro',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_svm.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_svm.best_params_}")
best_svm = grid_svm.best_estimator_

In [ ]:
y_pred_best = grid_svm.predict(X_test)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best))

# (DESCARTADO) MODELO DE CLASIFICACIÓN CARACTERIZACION + ACTIVIDADES

## Con tres clases haciendo balance en Random Forest

In [ ]:
df_final_for_model.columns

In [ ]:
df_final_for_model.head(16)

### Un clasificador Random Forest

In [ ]:
exclude = [
    'username', 'platform',
    'grupo', 'peso' # Columnas de texto
]

X = df_final_for_model.drop(columns=exclude)
y = df_final_for_model['grupo']

X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=150, max_depth=12, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

Está inflado porque se repiten las filas.
Modelos lineales mixtos
MLM - variable de control (estudiante) - controlar el efecto aleatorio y le resta el efecto.

In [ ]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values(by='importance', ascending=False)

print("Top 15 variables más importantes:")
print(importances.head(15))

In [ ]:


y_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=rf.classes_,
            yticklabels=rf.classes_)

plt.title('Matriz de Confusión - Modelo de Predicción Temprana')
plt.ylabel('Clase Real')
plt.xlabel('Clase Predicha')

# Guardar o mostrar
plt.savefig('matriz_confusion_final.png')
plt.show()

### Búsqueda de hiperparámetros para Random Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', 'balanced_subsample']
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_

In [ ]:
from sklearn.metrics import classification_report

y_pred_best = best_rf.predict(X_test)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best))

### Logistic Regression y XGBoost

In [ ]:

# re-etiquetar para xgboost
target_map = {val: i for i, val in enumerate(rf.classes_)}
y_train_num = y_train.map(target_map)
y_test_num = y_test.map(target_map)

models = {
    "Random Forest": rf,
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
}

results = []

for name, model in models.items():
    if name == "XGBoost":
        target_train = y_train_num.values # valores mapeados
        target_test = y_test_num.values
    else:
        target_train = y_train.values
        target_test = y_test.values

    if name != "Random Forest":
        model.fit(X_train.values, target_train)

    y_pred = model.predict(X_test.values)

    acc = accuracy_score(target_test, y_pred)
    f1 = f1_score(target_test, y_pred, average='macro')

    recalls = recall_score(target_test, y_pred, average=None)
    rec_desertor = recalls[target_map['desertores']]

    results.append({
        "Modelo": name,
        "Accuracy": acc,
        "F1-Score (Macro)": f1,
        "Recall Desertores": rec_desertor
    })

df_comparacion = pd.DataFrame(results)
print(df_comparacion.sort_values(by="F1-Score (Macro)", ascending=False))

## Con tres clases balanceadas SMOTE

In [ ]:
exclude = ['grupo', 'username', 'platform', 'peso']
X = df_final_for_model.drop(columns=exclude)
y = df_final_for_model['grupo']

X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

# Split Estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Aplicar SMOTE al set de entrenamiento
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Distribución original: {y_train.value_counts().to_dict()}")
print(f"Distribución con SMOTE: {y_train_res.value_counts().to_dict()}")


### Búsqueda de hiperparámetros Random Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': [None]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_res, y_train_res)

print(f"Mejores parámetros: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_

In [ ]:
y_pred_best = best_rf.predict(X_test)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best))

In [ ]:
importancias_char = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importancias_char.head(10), palette='magma')

plt.title('Top 10 Predictores: Caracterización Inicial (Día 0)', fontsize=14)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable de la Encuesta', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

print(importancias_char.head(10))

### (No corre) Búsqueda de hiperparámetros para SVM

In [ ]:
print(f"Dimensiones: {X_train_res.shape}")
print(f"Tipos de datos:\n{X_train_res.dtypes.value_counts()}")

In [ ]:
print(f"¿Hay valores nulos?: {X_train_res.isnull().sum().sum()}")
print(f"¿Hay valores infinitos?: {np.isinf(X_train_res).values.sum()}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# booleanos a enteros (0 y 1)
X_train_svm = X_train_res.astype(float)
X_test_svm = X_test.astype(float)

#  escalador
scaler = StandardScaler()

# escalar los datos
#
X_train_scaled = scaler.fit_transform(X_train_svm)

# Solo transform en el set de test (usando la media y std del entrenamiento)
X_test_scaled = scaler.transform(X_test_svm)

print("Datos escalados correctamente. Los booleanos ahora son decimales.")

In [ ]:
# Definimos el espacio de búsqueda
# C: Penalización del error (regularización)
# gamma: Define qué tanto influye un solo ejemplo de entrenamiento
# kernel: La función que transforma los datos a un espacio de mayor dimensión
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'poly', 'sigmoid'],
    'class_weight': [None]
}

grid_svm = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=param_grid_svm,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_svm.fit(X_train_scaled, y_train_res)

print(f"Mejores parámetros SVM: {grid_svm.best_params_}")
best_svm = grid_svm.best_estimator_

### LinearSVC

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

param_grid_fast = {
    'C': [0.01, 0.1, 1, 10],
    'class_weight': [None]
}

grid_svm_fast = GridSearchCV(
    estimator=LinearSVC(dual=False, random_state=42, max_iter=2000),
    param_grid=param_grid_fast,
    scoring='recall_macro',
    cv=3,
    verbose=2
)

grid_svm_fast.fit(X_train_scaled, y_train_res)

print(f"Mejores parámetros: {grid_svm_fast.best_params_}")
best_svm = grid_svm_fast.best_estimator_

In [ ]:
y_pred_best_svm = grid_svm_fast.predict(X_test_scaled)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best_svm))

## Con dos clases

In [ ]:
# Aprobados y Reprobados se agrupan
df_final_for_model['grupo_binario'] = df_final_for_model['grupo'].replace({
    'desertores': 'reprobados',
    'reprobados': 'reprobados'
})


In [ ]:
exclude = [
    'username', 'platform',
    'grupo', 'peso', 'grupo_binario'
]

X = df_final_for_model.drop(columns=exclude)
y = df_final_for_model['grupo_binario']

X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


### Búsqueda de hiperparámetros Random Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': [None]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_search.best_params_}")
best_rf = grid_search.best_estimator_

In [ ]:
y_pred_best = best_rf.predict(X_test)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best))

In [ ]:
importancias_char = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importancias_char.head(10), palette='magma')

plt.title('Top 10 Predictores: Caracterización Inicial (Día 0)', fontsize=14)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable de la Encuesta', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

print(importancias_char.head(10))

### (No corre) Búsqueda de hiperparámetros para SVM

In [ ]:
from sklearn.preprocessing import StandardScaler

# booleanos a enteros (0 y 1)
X_train_svm = X_train.astype(float)
X_test_svm = X_test.astype(float)

#  escalador
scaler = StandardScaler()

# escalar los datos
#
X_train_scaled = scaler.fit_transform(X_train_svm)

# Solo transform en el set de test (usando la media y std del entrenamiento)
X_test_scaled = scaler.transform(X_test_svm)

print("Datos escalados correctamente. Los booleanos ahora son decimales.")

In [ ]:
# Definimos el espacio de búsqueda
# C: Penalización del error (regularización)
# gamma: Define qué tanto influye un solo ejemplo de entrenamiento
# kernel: La función que transforma los datos a un espacio de mayor dimensión
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': [1, 0.1, 0.01, 0.001],
    'kernel': ['rbf', 'poly', 'sigmoid'],
    'class_weight': [None]
}

grid_svm = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=param_grid_svm,
    scoring='recall_macro',
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_svm.fit(X_train_scaled, y_train_res)

print(f"Mejores parámetros SVM: {grid_svm.best_params_}")
best_svm = grid_svm.best_estimator_

### LinearSVC

In [ ]:
X_train_scaled

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

param_grid_fast = {
    'C': [0.01, 0.1, 1, 10],
    'class_weight': [None]
}

grid_svm_fast = GridSearchCV(
    estimator=LinearSVC(dual=False, random_state=42, max_iter=2000),
    param_grid=param_grid_fast,
    scoring='recall_macro',
    cv=3,
    verbose=2
)

grid_svm_fast.fit(X_train_scaled, y_train)

print(f"Mejores parámetros: {grid_svm_fast.best_params_}")
best_svm = grid_svm_fast.best_estimator_

In [ ]:
y_pred_best_svm = grid_svm_fast.predict(X_test_scaled)
print("--- RESULTADOS DEL MODELO OPTIMIZADO ---")
print(classification_report(y_test, y_pred_best_svm))

# MODELO DE ANÁLISIS DE SUPERVIVENCIA

In [ ]:
print(df_final_for_model.columns)



>

      ['username', 'platform',
      # columnas de actividad
       'actividad', 'submission_activity_user',
       'max_grade', 'custom[custom_summary_result]', 'grade', 'peso', 'result',
       'status', 'best', 'dias_totales',
        ## caracterizacion por estudiante
       'conteo_lenguajes',
       'lenguajes_programacion',
       'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql',
       # medio
       'medio_canal_ads_social',
       'medio_canal_busqueda', 'medio_canal_push_notif',
       'medio_canal_social_directo', 'medio_campo_corregido',
       'medio_declaro_sin_experiencia', 'medio_es_spam_ruido',
       'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos',
       # motivacion
       'mot_data_science_ia',
       'mot_declaro_sin_experiencia', 'mot_dev_backend', 'mot_dev_web',
       'mot_es_spam_ruido', 'mot_mot_aprendizaje_general',
       'mot_mot_ciencia_datos', 'mot_mot_desarrollo', 'mot_mot_profesional',
       'mot_no_reporta', 'mot_otras_motivaciones_especificas',
       'mot_principiante_general',
       # ocupacion
       'ocupacion_empleado',
       'ocupacion_es_spam_ruido', 'ocupacion_estudiante',
       'ocupacion_independiente', 'ocupacion_no_reporta',
       'ocupacion_otros_perfiles_ocupacion', 'ocupacion_pensionado',
       'ocupacion_perfil_busqueda_empleo', 'ocupacion_perfil_educador',
       'ocupacion_perfil_estudia_trabaja', 'ocupacion_tareas_del_hogar',
       
       'dedicacion_num', 'nivel_num',
        ## clase a predecir
       'grupo'],

In [ ]:
df_model_tasks = df_final_for_model[
       ['username', 'platform', 'actividad', 'submission_activity_user',
       #'max_grade',
        'custom[custom_summary_result]', 'grade',
        #'peso',
       'result',
       'status',
        #'best',
        'dias_x_actividad',
        'conteo_lenguajes',
       'lenguajes_programacion', 'grupo', 'otro_lenguaje_campo_corregido',
       'otro_lenguaje_declaro_sin_experiencia', 'otro_lenguaje_es_spam_ruido',
       'otro_lenguaje_no_reporta', 'otro_lenguaje_otros_lenguajes_especificos',
       'otro_lenguaje_pref_academico', 'otro_lenguaje_pref_bi_datos',
       'otro_lenguaje_pref_legacy', 'otro_lenguaje_pref_tecnico_otros',
       'otro_lenguaje_pref_web_sql', 'medio_canal_ads_social',
       'medio_canal_busqueda', 'medio_canal_push_notif',
       'medio_canal_social_directo', 'medio_campo_corregido',
       'medio_declaro_sin_experiencia', 'medio_es_spam_ruido',
       'medio_no_reporta', 'medio_origen_academico',
       'medio_origen_redes_sociales', 'medio_origen_web_busqueda',
       'medio_otros_medios_especificos', 'mot_data_science_ia',
       'mot_declaro_sin_experiencia', 'mot_dev_backend', 'mot_dev_web',
       'mot_es_spam_ruido', 'mot_mot_aprendizaje_general',
       'mot_mot_ciencia_datos', 'mot_mot_desarrollo', 'mot_mot_profesional',
       'mot_no_reporta', 'mot_otras_motivaciones_especificas',
       'mot_principiante_general', 'ocupacion_empleado',
       'ocupacion_es_spam_ruido', 'ocupacion_estudiante',
       'ocupacion_independiente', 'ocupacion_no_reporta',
       'ocupacion_otros_perfiles_ocupacion', 'ocupacion_pensionado',
       'ocupacion_perfil_busqueda_empleo', 'ocupacion_perfil_educador',
       'ocupacion_perfil_estudia_trabaja', 'ocupacion_tareas_del_hogar',
       'dedicacion_num', 'nivel_num']]

In [ ]:
df_model_tasks['prefijo'] = df_model_tasks['actividad'].str.extract(r'(a\d+)')


Se hace  un pivot que cree columnas por actividad

In [ ]:
df_model_tasks['custom[custom_summary_result]'].value_counts()

In [ ]:
df_model_tasks['status'].value_counts()

In [ ]:
df_model_tasks['is_error'] = (df_model_tasks['status'] == 'error').astype(int)

df_model_tasks['is_wrong_answer'] = (df_model_tasks['custom[custom_summary_result]'] == 'WRONG_ANSWER').astype(int)
df_model_tasks['is_runtime_error'] = (df_model_tasks['custom[custom_summary_result]'] == 'RUNTIME_ERROR').astype(int)

In [ ]:
metrics = ['grade', 'submission_activity_user',
                    'is_error', 'is_wrong_answer', 'is_runtime_error',
                    'dias_x_actividad'
                    ]


df_pivot_full = df_model_tasks.pivot_table(
    index='username',
    columns='prefijo',
    values=metrics,
    fill_value=0
)

df_pivot_full.columns = [f'{col[1]}_{col[0]}' for col in df_pivot_full.columns]
df_pivot_full = df_pivot_full.reset_index()

df_ultra_model = pd.merge(df_char_model_unique, df_pivot_full, on='username', how='inner')

In [ ]:

cols_notas = [f'a{i}_grade' for i in range(1, 18)]

def calcular_ultima_señal_vida(row):
    # Si el estudiante llegó al final del proceso (aprobado o reprobado)
    # asigna 0 o un valor que indique "No desertó"
    if row['grupo'] in ['aprobados', 'reprobados']:
        return 0

    # Para los desertores, se busca de atrás hacia adelante
    # la última tarea donde tuvo una nota mayor a 0
    for i in range(17, 0, -1):
        col = f'a{i}_grade'
        if col in row and row[col] > 0:
            return i

    return 1 # Si no tiene nada, desertó desde la primera

df_ultra_model['punto_quiebre_real'] = df_ultra_model.apply(calcular_ultima_señal_vida, axis=1)

In [ ]:
#  cuántas tareas entregó en total de las 17
#df_ultra_model['total_entregas'] = df_ultra_model[cols_notas].gt(0).sum(axis=1)

# la "densidad" de entrega (¿qué tan seguido entrega?)
#df_ultra_model['consistencia'] = df_ultra_model['total_entregas'] / 17

In [ ]:
print(df_ultra_model['grupo'].value_counts())

In [ ]:
columnas_total_actividad = [c for c in df_ultra_model.columns if re.match(r'^a\d+_', c)]
columnas_carac = [c for c in df_char_model_unique.columns if c not in ['username', 'platform', 'grupo', 'grupo_binario']]

## Tres clases

In [ ]:
X = df_ultra_model[columnas_carac + columnas_total_actividad]
y = df_ultra_model['grupo']

In [ ]:
df_ultra_model['grupo'].value_counts()

El dataset tiene un desbalance de clases.

Si se hace un split (80/20) y se buscan hiperparámetros solo en el 80%, se corre el riesgo de que en ese fragmento haya tan pocos "reprobados" que el RandomizedSearchCV elija parámetros que ignoren por completo a ese grupo.

Al usar todo el dataset para la búsqueda inicial, se asegura que los hiperparámetros (como class_weight='balanced') se calibren con la proporción real de alumnos del curso.

Splittear
**nested cross-validation para probar**

Si se hace una búsqueda de hiperparámetros dentro de cada una de las 17 tareas:

En la Tarea 1, el buscador podría elegir un max_depth=3 (porque hay pocos datos).

En la Tarea 17, podría elegir un max_depth=15.

El problema: La gráfica de F1-Score ya no mostraría la evolución de los datos, sino la evolución de los parámetros. No se sabría si el F1 subió porque la Tarea 17 es mejor o porque el modelo se volvió más complejo.

### Experimentación 1 = busqueda de hiperparámetros en un solo modelo

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 8, 12, 15],
    'min_samples_leaf': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', 'balanced_subsample']
}

rs = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42
)

rs.fit(X, y) ### sobre todo el dataset para encontrar la mejor combinación
rf_optimo = rs.best_estimator_

print(f"Mejores parámetros encontrados: {rs.best_params_}")

En qué momento del curso más temprano el modelo puede predecir

In [ ]:
import re

columnas_total_actividad = [c for c in df_ultra_model.columns if re.match(r'^a\d+_', c)]
resultados_progreso = []
max_tareas = 17

for n in range(max_tareas + 1):
    if n == 0:
        cols_sel = columnas_carac
    else:
        filtro_tareas = [f'a{i}_' for i in range(1, n+1)]
        cols_actividad = [c for c in columnas_total_actividad if any(c.startswith(t) for t in filtro_tareas)]
        cols_sel = columnas_carac + cols_actividad

    scores = cross_val_score(rf_optimo, df_ultra_model[cols_sel], y, cv=5, scoring='f1_macro')

    resultados_progreso.append({
        'n_tareas': n,
        'f1_macro': scores.mean(),
        'error': scores.std()
    })

df_evolucion = pd.DataFrame(resultados_progreso)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df_evolucion['n_tareas'], df_evolucion['f1_macro'], color='#5D3FD3', marker='o', linewidth=2)

plt.fill_between(
    df_evolucion['n_tareas'],
    df_evolucion['f1_macro'] - df_evolucion['error'],
    df_evolucion['f1_macro'] + df_evolucion['error'],
    color='#5D3FD3', alpha=0.1
)

plt.title('Poder Predictivo Acumulado: De la Tarea 1 a la 17', fontsize=14)
plt.xlabel('Número de Tareas Incluidas en el Modelo')
plt.ylabel('F1-Score Macro (Promedio CV)')
plt.xticks(range(0, 18))
plt.grid(True, linestyle='--', alpha=0.5)


plt.show()

In [ ]:
importances = rf_optimo.feature_importances_
feature_names = cols_sel

df_importancia = pd.DataFrame({'Variable': feature_names, 'Importancia': importances})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=df_importancia, palette='viridis')
plt.title('Top 15 Variables que Predicen el Éxito/Deserción')
plt.show()

Para tener el detalle del f1 por clase:

In [ ]:
from sklearn.model_selection import cross_val_predict


resultados_por_grupo = []

for n in range(max_tareas + 1):
    if n == 0:
        cols_sel = columnas_carac
    else:
        filtro_tareas = [f'a{i}_' for i in range(1, n+1)]
        cols_sel = columnas_carac + [c for c in X.columns if any(c.startswith(t) for t in filtro_tareas)]


    y_pred_cv = cross_val_predict(rf_optimo, X[cols_sel], y, cv=5)

    f1_por_clase = f1_score(y, y_pred_cv, average=None, labels=rf_optimo.classes_)

    res = {'n_tareas': n}
    for i, clase in enumerate(rf_optimo.classes_):
        res[clase] = f1_por_clase[i]

    resultados_por_grupo.append(res)

df_grupos_cv = pd.DataFrame(resultados_por_grupo)

In [ ]:
plt.figure(figsize=(12, 6))

colores = {
    'aprobados': '#2ecc71',
    'desertores': '#f1c40f',
    'reprobados': '#e74c3c'
}

for clase in rf_optimo.classes_:
    plt.plot(df_grupos_cv['n_tareas'], df_grupos_cv[clase],
             marker='o', label=f'F1: {clase}', color=colores.get(clase), linewidth=2)

plt.title('Evolución del F1-Score por Grupo (Validación Cruzada)', fontsize=14)
plt.xlabel('Número de Tareas Observadas')
plt.ylabel('F1-Score')
plt.xticks(range(0, 18))
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### Búsqueda de hiperparámetros para cada tiempo-evento

Búsqueda de hiperparámetos para cada tiempo-evento

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import re

param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', 'balanced_subsample']
}

X_train_full, X_test_full, y_train, y_test = train_test_split(
    df_ultra_model, y, test_size=0.2, stratify=y, random_state=42
)

resultados_progreso = []
max_tareas = 17

for n in range(max_tareas + 1):
    if n == 0:
        cols_sel = columnas_carac
    else:
        filtro_tareas = [f'a{i}_' for i in range(1, n+1)]
        cols_actividad = [c for c in columnas_total_actividad if any(c.startswith(t) for t in filtro_tareas)]
        cols_sel = columnas_carac + cols_actividad

    X_train_n = X_train_full[cols_sel]
    X_test_n = X_test_full[cols_sel]

    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42),
        param_distributions=param_dist,
        n_iter=10,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train_n, y_train)

    best_model = search.best_estimator_
    y_pred_test = best_model.predict(X_test_n)
    f1_test = f1_score(y_test, y_pred_test, average='macro')

    resultados_progreso.append({
        'n_tareas': n,
        'f1_train_cv': search.best_score_,
        'f1_test_final': f1_test,
        'n_features': len(cols_sel)
    })

    print(f"Tarea {n}: Train F1={search.best_score_:.3f} | Test F1={f1_test:.3f}")

df_evolucion = pd.DataFrame(resultados_progreso)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(df_evolucion['n_tareas'], df_evolucion['f1_train_cv'],
         label='Entrenamiento (CV)', color='#A9A9A9', linestyle='--', alpha=0.7)

plt.plot(df_evolucion['n_tareas'], df_evolucion['f1_test_final'],
         label='Test Final (Datos nuevos)', color='#5D3FD3', marker='o', linewidth=2.5)

plt.fill_between(
    df_evolucion['n_tareas'],
    df_evolucion['f1_train_cv'],
    df_evolucion['f1_test_final'],
    color='#5D3FD3', alpha=0.1, label='Brecha de Generalización'
)

plt.title('Poder Predictivo Acumulado: Evolución del Modelo Tarea tras Tarea', fontsize=14, pad=20)
plt.xlabel('Número de Tareas Incluidas en el Modelo', fontsize=12)
plt.ylabel('F1-Score Macro', fontsize=12)
plt.xticks(range(0, 18))
plt.ylim(0, 1.0)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower right')


plt.tight_layout()
plt.show()

Se está balanceando las clases en el mismo random forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

etiquetas = ['aprobados', 'desertores', 'reprobados']

X_train_full, X_test_full, y_train, y_test = train_test_split(
    df_ultra_model, y, test_size=0.2, stratify=y, random_state=42
)

param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', 'balanced_subsample']
}

resultados_progreso = []

for n in range(max_tareas + 1):
    if n == 0:
        cols_sel = columnas_carac
    else:
        filtro_tareas = [f'a{i}_' for i in range(1, n+1)]
        cols_actividad = [c for c in columnas_total_actividad if any(c.startswith(t) for t in filtro_tareas)]
        cols_sel = columnas_carac + cols_actividad

    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42),
        param_distributions=param_dist,
        n_iter=8,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train_full[cols_sel], y_train)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test_full[cols_sel])

    f1_vals = f1_score(y_test, y_pred, average=None, labels=etiquetas, zero_division=0)

    resultados_progreso.append({
        'n_tareas': n,
        'f1_aprobados': f1_vals[0],
        'f1_desertores': f1_vals[1],
        'f1_reprobados': f1_vals[2],
        'best_params': search.best_params_
    })

    print(f"Tarea {n} finalizada con éxito.")

df_evolucion_3_opt = pd.DataFrame(resultados_progreso)

In [ ]:
plt.figure(figsize=(14, 7))

colores = {'aprobados': '#2ecc71', 'desertores': '#f39c12', 'reprobados': '#e74c3c'}

for clase, color in colores.items():
    plt.plot(df_evolucion_3_opt['n_tareas'],
             df_evolucion_3_opt[f'f1_{clase}'],
             label=clase.capitalize(),
             color=color,
             marker='o',
             linewidth=2)

plt.title('Evolución del F1-Score por Grupo (Con Optimización de Hiperparámetros)', fontsize=15)
plt.xlabel('Número de Tareas Incluidas', fontsize=12)
plt.ylabel('F1-Score (Test Set)', fontsize=12)
plt.xticks(range(0, 18))
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Grupos", loc='upper left')

plt.tight_layout()
plt.show()

TODO: hacer lo mismo para las dos clases
- XGBOOST
- SVM
- Random Forest




## Dos clases

In [ ]:
# Aprobados y Reprobados se agrupan
df_ultra_model['grupo_binario'] = df_ultra_model['grupo'].replace({
    'desertores': 'reprobados',
    'reprobados': 'reprobados'
})


In [ ]:
X = df_ultra_model[columnas_carac + columnas_total_actividad]
y = df_ultra_model['grupo_binario']

### Búsqueda de hiperparámetros para cada tiempo-evento

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import re

param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', 'balanced_subsample']
}

X_train_full, X_test_full, y_train, y_test = train_test_split(
    df_ultra_model, y, test_size=0.2, stratify=y, random_state=42
)

resultados_progreso = []
max_tareas = 17

for n in range(max_tareas + 1):
    if n == 0:
        cols_sel = columnas_carac
    else:
        filtro_tareas = [f'a{i}_' for i in range(1, n+1)]
        cols_actividad = [c for c in columnas_total_actividad if any(c.startswith(t) for t in filtro_tareas)]
        cols_sel = columnas_carac + cols_actividad

    X_train_n = X_train_full[cols_sel]
    X_test_n = X_test_full[cols_sel]

    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42),
        param_distributions=param_dist,
        n_iter=10,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train_n, y_train)

    best_model = search.best_estimator_
    y_pred_test = best_model.predict(X_test_n)
    f1_test = f1_score(y_test, y_pred_test, average='macro')

    resultados_progreso.append({
        'n_tareas': n,
        'f1_train_cv': search.best_score_,
        'f1_test_final': f1_test,
        'n_features': len(cols_sel)
    })

    print(f"Tarea {n}: Train F1={search.best_score_:.3f} | Test F1={f1_test:.3f}")

df_evolucion = pd.DataFrame(resultados_progreso)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(df_evolucion['n_tareas'], df_evolucion['f1_train_cv'],
         label='Entrenamiento (CV)', color='#A9A9A9', linestyle='--', alpha=0.7)

plt.plot(df_evolucion['n_tareas'], df_evolucion['f1_test_final'],
         label='Test Final (Datos nuevos)', color='#5D3FD3', marker='o', linewidth=2.5)

plt.fill_between(
    df_evolucion['n_tareas'],
    df_evolucion['f1_train_cv'],
    df_evolucion['f1_test_final'],
    color='#5D3FD3', alpha=0.1, label='Brecha de Generalización'
)

plt.title('Poder Predictivo Acumulado: Evolución del Modelo Tarea tras Tarea', fontsize=14, pad=20)
plt.xlabel('Número de Tareas Incluidas en el Modelo', fontsize=12)
plt.ylabel('F1-Score Macro', fontsize=12)
plt.xticks(range(0, 18))
plt.ylim(0, 1.0)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower right')


plt.tight_layout()
plt.show()

modelo por clase:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

etiquetas = ['aprobados', 'reprobados']

X_train_full, X_test_full, y_train, y_test = train_test_split(
    df_ultra_model, y, test_size=0.2, stratify=y, random_state=42
)

param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', 'balanced_subsample']
}

resultados_progreso = []

for n in range(max_tareas + 1):
    if n == 0:
        cols_sel = columnas_carac
    else:
        filtro_tareas = [f'a{i}_' for i in range(1, n+1)]
        cols_actividad = [c for c in columnas_total_actividad if any(c.startswith(t) for t in filtro_tareas)]
        cols_sel = columnas_carac + cols_actividad

    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42),
        param_distributions=param_dist,
        n_iter=8,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train_full[cols_sel], y_train)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test_full[cols_sel])


    resultados_progreso.append({
        'n_tareas': n,
        'f1_aprobados': f1_vals[0],
        'f1_reprobados': f1_vals[1],
        'best_params': search.best_params_
    })

    print(f"Tarea {n} finalizada con éxito.")

df_evolucion_3_opt = pd.DataFrame(resultados_progreso)

In [ ]:
plt.figure(figsize=(14, 7))

colores = {'aprobados': '#2ecc71',  'reprobados': '#e74c3c'}

for clase, color in colores.items():
    plt.plot(df_evolucion_3_opt['n_tareas'],
             df_evolucion_3_opt[f'f1_{clase}'],
             label=clase.capitalize(),
             color=color,
             marker='o',
             linewidth=2)

plt.title('Evolución del F1-Score por Grupo (Con Optimización de Hiperparámetros)', fontsize=15)
plt.xlabel('Número de Tareas Incluidas', fontsize=12)
plt.ylabel('F1-Score (Test Set)', fontsize=12)
plt.xticks(range(0, 18))
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Grupos", loc='upper left')

plt.tight_layout()
plt.show()